In [2]:
import os
from phoenix.otel import register

# Add Phoenix API Key for tracing
PHOENIX_API_KEY = os.getenv("PHOENIX_API_KEY")
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={PHOENIX_API_KEY}"

# configure the Phoenix tracer
tracer_provider = register(
  endpoint="https://app.phoenix.arize.com/v1/traces",
) 

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/v1/traces
|  Transport: HTTP
|  Transport Headers: {'api_key': '****', 'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [3]:
from openinference.instrumentation.openai import OpenAIInstrumentor

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

In [4]:
from pydantic_ai import Agent

agent = Agent(  
    'openai:gpt-4o',
    system_prompt='Be concise, reply with one sentence.',  
)

result = await agent.run('How many feet are in a mile?')  
print(result.data)

Failed to export batch code: 204, reason: 


There are 5,280 feet in a mile.


In [5]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
from dataclasses import dataclass

from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext


@dataclass
class NameContext:
    """Dependencies (context) for the conversation."""
    user_name: str


class GreetingResult(BaseModel):
    """Structured output from the AI."""
    greeting: str = Field(description="A short greeting to the user")


greeting_agent = Agent(
    model="openai:gpt-4o",
    deps_type=NameContext,
    result_type=GreetingResult,
    system_prompt=(
        "You are a personalized greeter AI. "
        "Return a short greeting for the user."
    ),
)


@greeting_agent.system_prompt
async def add_user_name(ctx: RunContext[NameContext]) -> str:
    return f"The user's name is {ctx.deps.user_name!r}."


async def main():
    deps = NameContext(user_name="Alice")

    result = await greeting_agent.run(
        "Hi, can you greet me?",
        deps=deps
    )

    print(result.data)

await main()

Failed to export batch code: 204, reason: 


greeting="Hello, Alice! Hope you're having a wonderful day!"


```
{"messages": [{"role": "system", "content": "You are a personalized greeter AI. Return a short greeting for the user."}, {"role": "system", "content": "The user's name is 'Alice'."}, {"role": "user", "content": "Hi, can you greet me?"}], "model": "gpt-4o", "n": 1, "parallel_tool_calls": true, "stream": false, "tool_choice": "required", "tools": [{"type": "function", "function": {"name": "final_result", "description": "Structured output from the AI.", "parameters": {"properties": {"greeting": {"description": "A short greeting to the user", "title": "Greeting", "type": "string"}}, "required": ["greeting"], "title": "GreetingResult", "type": "object"}}}]}
```

In [7]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import json
from dataclasses import dataclass
from typing import List

from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext


@dataclass
class ProblemContext:
    user_name: str


class DatabaseQueryAnalysis(BaseModel):
    queries: List[str] = Field(default_factory=list)


analysis_agent = Agent(
    model="openai:gpt-4o",
    deps_type=ProblemContext,
    result_type=DatabaseQueryAnalysis,
    system_prompt=(
        "You are an AI that, given a user's problem, identifies what database queries "
        "would be needed to retrieve information that solves the user's problem."
    ),
)


@analysis_agent.system_prompt
async def analysis_agent_system_prompt(ctx: RunContext[ProblemContext]) -> str:
    return (
        f"The user's name is {ctx.deps.user_name!r}. "
        "Analyze the user's input and suggest relevant database queries."
    )


class QueryAPIRequest(BaseModel):
    query_text: str = Field(description="The raw query to be executed.")
    endpoint: str = Field(
        default="/execute-query",
        description="The endpoint where the query should be sent."
    )


class QueryAPIRequests(BaseModel):
    requests: List[QueryAPIRequest] = Field(default_factory=list)


formatting_agent = Agent(
    model="openai:gpt-4o",
    deps_type=ProblemContext,
    result_type=QueryAPIRequests,
    system_prompt=(
        "You are an AI that formats database queries into the provided Pydantic BaseModels for API requests."
    ),
)


@formatting_agent.system_prompt
async def formatting_agent_prompt(ctx: RunContext[ProblemContext]) -> str:
    return (
        f"The user's name is {ctx.deps.user_name!r}. "
        "Convert the list of database queries into `QueryAPIRequest` objects, "
        "wrapped in the `QueryAPIRequests` model."
    )


async def main():
    deps = ProblemContext(user_name="Alice")

    user_problem = (
        "I need to find the top-rated restaurants near me. "
        "Show me some options sorted by rating and distance."
    )

    print("[Step 1] Analyzing the user's problem to find relevant database queries...")
    analysis_result = await analysis_agent.run(user_problem, deps=deps)

    # Convert to dict and JSON-serialize with indentation
    print("Analysis result (queries):")
    print(json.dumps(analysis_result.data.model_dump(), indent=2))

    queries_to_format = analysis_result.data.queries
    second_input = (
        "Here are the queries to format:\n" + "\n".join(queries_to_format)
    )

    print("\n[Step 2] Formatting the queries into Pydantic API requests...")
    formatting_result = await formatting_agent.run(second_input, deps=deps)

    # Again, convert to dict and JSON-serialize with indentation
    print("Formatting result (API requests):")
    print(json.dumps(formatting_result.data.model_dump(), indent=2))


# If you're in an environment that supports top-level await (e.g., Jupyter):
await main()

# If you're in a standard Python script:
# if __name__ == "__main__":
#     asyncio.run(main())


[Step 1] Analyzing the user's problem to find relevant database queries...


Failed to export batch code: 204, reason: 


Analysis result (queries):
{
  "queries": [
    "SELECT name, rating, distance FROM restaurants WHERE location = 'current location' ORDER BY rating DESC, distance ASC;"
  ]
}

[Step 2] Formatting the queries into Pydantic API requests...


Failed to export batch code: 204, reason: 


Formatting result (API requests):
{
  "requests": [
    {
      "query_text": "SELECT name, rating, distance FROM restaurants WHERE location = 'current location' ORDER BY rating DESC, distance ASC;",
      "endpoint": "/execute-query"
    }
  ]
}
